### Configuración inicial

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
from delta import *
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, TimestampType


builder = SparkSession.builder \
    .appName("Lab_SECOP_Silver") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f9d4c3d7-3228-4c93-b942-5ffd2c007865;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 289ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

### Leer datos de Bronce

In [ ]:
bronze_path = "/app/data/lakehouse/bronze/secop"
df_bronze = spark.read.format("delta").load(bronze_path)

26/01/30 22:25:33 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/01/30 22:25:48 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/01/30 22:26:03 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


In [ ]:
df_bronze.limit(5).toPandas()

In [ ]:
# 2. Tipado inicial y Renombrado

df_transform = df_bronze \
    .withColumn("precio_base_num", F.col("valor_del_contrato").cast(DecimalType(18, 2))) \
    .withColumn("fecha_firma_dt", F.to_date(F.col("fecha_de_firma")))

# 3. Definir Reglas de Calidad (Quality Gate)
# Regla: Precio > 0 Y Fecha no nula
condicion_valida = (F.col("precio_base_num") > 0) & (F.col("fecha_firma_dt").isNotNull())